In [ ]:
# !pip install openai==2.14.0
# !pip install pandas
# !pip install datasets

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:

import openai
import pandas as pd

In [5]:
path = "datasets/qald.json"

In [6]:



if path.split('.')[-1] =='json':
  data = pd.read_json(path)
else:
  data = pd.read_csv(path)


unwanted ={'mintaka':['id', 'translations', 'questionEntity', 'category', 'complexityType', 'answer'], 'qald': None,'hotpot':['supporting_facts', 'level', 'context', 'answer', '_id','type']}
#Create a dictionary that stores the name of the dataset as the key, and the list of unwanted columns as a key, and then call data.drop(columns = unwanted['dataset_name'])

pre =path.split('/')
name = pre[-1].split('.')[0]


if name in unwanted.keys():
  cols = unwanted[name]


if name =='qald':
  questions = []
  for index, row in data.iterrows():
    question = row['questions']['question'][0]['string']
    questions.append(question)
  data = pd.DataFrame({'question':questions})


if cols !=None:
  data.drop(columns = cols,inplace=True)

data = data[:100]

In [7]:
data.head()

,question
0,What is the time zone of Salt Lake City?
1,Who killed Caesar?
2,What is the highest mountain in Germany?
3,Which American presidents were in office durin...
4,Butch Otter is the governor of which U.S. state?


In [8]:
data

,question
0,What is the time zone of Salt Lake City?
1,Who killed Caesar?
2,What is the highest mountain in Germany?
3,Which American presidents were in office durin...
4,Butch Otter is the governor of which U.S. state?
...,...
95,Where does Piccadilly start?
96,What is the name of the university where Obama...
97,When did Paraguay proclaim its independence?
98,How short is the shortest active NBA player?


In [9]:
## Mintaka
from openai import OpenAI
from dotenv import load_dotenv

openai_key = load_dotenv('OPENAI_API_KEY')
client = OpenAI(api_key=openai_key)

# Few-shot examples for translation to AAVE
few_shot_examples = [
    "I was bewildered, but I knew dat it was no gud asking his ass to explain.",
    "Cochran pontificated windily for da camera.",
    "I don’t want them to follow in my footsteps, as I ain’t go to no college, but I want them to go."
]

#Translation API Call + Prompt

def translate_to_aave(sae_text, few_shot_examples):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that translates Standard American English (SAE) to African American Vernacular English (AAVE)."},
        {"role": "user", "content": (
            "Translate the following sentence from Standard English to African American Vernacular English (AAVE). "
            "Ensure the translation maintains the structure of the original sentence without adding extra information.\n\n"
            "Examples for reference:\n"
            "1. I was bewildered, but I knew dat it was no gud asking his ass to explain.\n"
            "2. Cochran pontificated windily for da camera.\n"
            "3. I don’t want them to follow in my footsteps, as I ain’t go to no college, but I want them to go.\n\n"

            f"Text: {sae_text}\n\n"
            "AAVE Translation:"
        )}
    ]

# Model
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        max_tokens=150,
        temperature=0.7
    )

    return response.choices[0].message.content.strip()



# What columns are being made in what order
translated_data = pd.DataFrame(columns=['AAVE Question'])


In [11]:

for index, row in to_translate.iterrows():
    sae_question = row['Question']
    aave_question = translate_to_aave(sae_question, few_shot_examples)

    print(f"Processed row {index + 1} question")



    # Creating new dataframe from results for csv file
    translated_data = pd.concat([translated_data, pd.DataFrame({
        'AAVE Question': [aave_question]
    })], ignore_index=True)




NameError: name 'to_translate' is not defined

In [ ]:
translated_data.head()

In [ ]:
translated_file_path = '/content/drive/Shareddrives/Algoverse_KSAC/AAVENUE translations/Translated_LC_better.csv'
translated_data.to_csv(translated_file_path, index=False)

print(f"Translations completed and saved to {translated_file_path}")

In [ ]:
## SimpleQuestions

import openai
import pandas as pd

# Set up your OpenAI API key
openai.api_key = ""  # Replace with your actual OpenAI API key

# Load the CSV file for COPA
csv_file_path = '/content/drive/MyDrive/Algoverse/Old Results/AAVE Translations/GPT 4.0 Translations, 4.0 Evaluations/Copa/Copa_superglue_500_4.0.csv'
data = pd.read_csv(csv_file_path)

# Load the CSV file for Actual Answer
actual_answer_csv_path = '/content/drive/MyDrive/Algoverse/New Results/Evaluation Results/GPT-4o Eval Results/COPA/COPA_Evaluation_GPT4o.csv'
actual_answer_data = pd.read_csv(actual_answer_csv_path)

# Few-shot examples for translation to AAVE
few_shot_examples = [
    "I was bewildered, but I knew dat it was no gud asking his ass to explain.",
    "Cochran pontificated windily for da camera.",
    "I don’t want them to follow in my footsteps, as I ain’t go to no college, but I want them to go."
]

# Translation API Call + Prompt
def translate_to_aave(sae_text, few_shot_examples):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that translates Standard American English (SAE) to African American Vernacular English (AAVE)."},
        {"role": "user", "content": (
            "Translate the following sentence from Standard English to African American Vernacular English (AAVE). "
            "Ensure the translation maintains the structure of the original sentence without adding extra information.\n\n"
            "Examples for reference:\n"
            "1. I was bewildered, but I knew dat it was no gud asking his ass to explain.\n"
            "2. Cochran pontificated windily for da camera.\n"
            "3. I don’t want them to follow in my footsteps, as I ain’t go to no college, but I want them to go.\n\n"
            f"Text: {sae_text}\n\n"
            "AAVE Translation:"
        )}
    ]

    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",  # Ensure the correct model is used
        messages=messages,
        max_tokens=150,
        temperature=0.7
    )

    return response.choices[0].message['content'].strip()

# Create a new DataFrame to store translations
translated_data = pd.DataFrame(columns=[
    'Premise', 'Choice 1', 'Choice 2',
    'Translated Premise', 'Translated Choice 1', 'Translated Choice 2', 'Actual Answer'
])

for index, row in data.iterrows():
    sae_premise = row['Premise']
    sae_choice1 = row['Choice1']
    sae_choice2 = row['Choice2']
    actual_answer = actual_answer_data.iloc[index]['Actual Answer']

    aave_premise = translate_to_aave(sae_premise, few_shot_examples)
    print(f"Processed row {index + 1} premise")

    aave_choice1 = translate_to_aave(sae_choice1, few_shot_examples)
    print(f"Processed row {index + 1} choice 1")

    aave_choice2 = translate_to_aave(sae_choice2, few_shot_examples)
    print(f"Processed row {index + 1} choice 2")

    # Creating new dataframe from results for csv file
    translated_data = pd.concat([translated_data, pd.DataFrame({
        'Premise': [sae_premise],
        'Choice 1': [sae_choice1],
        'Choice 2': [sae_choice2],
        'Translated Premise': [aave_premise],
        'Translated Choice 1': [aave_choice1],
        'Translated Choice 2': [aave_choice2],
        'Actual Answer': [actual_answer]
    })], ignore_index=True)

# Save translated results to CSV file
translated_csv_file_path = '/content/drive/MyDrive/Algoverse/New Results/GPT 4o mini Translations/Translated Copa.csv'
translated_data.to_csv(translated_csv_file_path, index=False)

print(f"Translations completed and saved to {translated_csv_file_path}")

In [ ]:
## HOTPOTQA

import openai
import pandas as pd

# Set your OpenAI API key
openai.api_key = ""  # Replace with your actual OpenAI API key

# Load the CSV file for MultiRC
csv_file_path = '/content/drive/MyDrive/Algoverse/Old Results/AAVE Translations/GPT 4.0 Translations, 4.0 Evaluations/Multi-RC/MultiRC_1000_4.0.csv'
data = pd.read_csv(csv_file_path)

# Few-shot examples for translation to AAVE
few_shot_examples = [
    "I was bewildered, but I knew dat it was no gud asking his ass to explain.",
    "Cochran pontificated windily for da camera.",
    "I don’t want them to follow in my footsteps, as I ain’t go to no college, but I want them to go."
]

#Translation API Call + Prompt

def translate_to_aave(sae_text, few_shot_examples):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that translates Standard American English (SAE) to African American Vernacular English (AAVE)."},
        {"role": "user", "content": (
            "Translate the following sentence from Standard English to African American Vernacular English (AAVE). "
            "Ensure the translation maintains the structure of the original sentence without adding extra information.\n\n"
            "Examples for reference:\n"
            "1. I was bewildered, but I knew dat it was no gud asking his ass to explain.\n"
            "2. Cochran pontificated windily for da camera.\n"
            "3. I don’t want them to follow in my footsteps, as I ain’t go to no college, but I want them to go.\n\n"
            f"Text: {sae_text}\n\n"
            "AAVE Translation:"
        )}
    ]

    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",  # Ensure the correct model is used
        messages=messages,
        max_tokens=200,
        temperature=0.7
    )

    return response.choices[0].message['content'].strip()

# Create a new DataFrame to store translations
translated_data = pd.DataFrame(columns=[
    'Paragraph', 'Question', 'Answer Choice',
    'Translated Paragraph', 'Translated Question', 'Translated Answer Choice',
    'Actual Label'
])

# Process each row for translation
for index, row in data.iterrows():
    sae_paragraph = row['Paragraph']
    sae_question = row['Question']
    sae_answer = row['Answer']
    actual_label = row['Actual Label']

    # Translate each part to AAVE
    aave_paragraph = translate_to_aave(sae_paragraph, few_shot_examples)
    print(f"Processed row {index + 1} paragraph")

    aave_question = translate_to_aave(sae_question, few_shot_examples)
    print(f"Processed row {index + 1} question")

    aave_answer = translate_to_aave(sae_answer, few_shot_examples)
    print(f"Processed row {index + 1} answer")

    # Creating new dataframe from results for csv file
    translated_data = pd.concat([translated_data, pd.DataFrame({
        'Paragraph': [sae_paragraph],
        'Question': [sae_question],
        'Answer Choice': [sae_answer],
        'Translated Paragraph': [aave_paragraph],
        'Translated Question': [aave_question],
        'Translated Answer Choice': [aave_answer],
        'Actual Label': [actual_label]
    })], ignore_index=True)

# Save translated results to CSV file
translated_csv_file_path = '/content/drive/MyDrive/Algoverse/New Results/GPT 4o mini Translations/Translated MultiRC.csv'
translated_data.to_csv(translated_csv_file_path, index=False)

# Print function
print(f"Translations completed and saved to {translated_csv_file_path}")